# Estudio Prospectivo de LLMs y ML en Predicción Financiera

## Trabajo Fin de Grado

Este notebook presenta el análisis completo del sistema híbrido ML + LLM para predicción de activos financieros y criptoactivos.

## 1. Configuración e Importaciones

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Importar módulos del sistema
from data.data_loader import FinancialDataLoader, CryptoDataLoader
from utils.data_utils import DataProcessor
from models.pattern_detection.chart_patterns import PatternDetector
from models.ml_models.traditional_ml import TraditionalMLModels, FeatureEngineer
from agents.technical_agent.technical_analyst import TechnicalAnalystAgent
from agents.sentiment_agent.sentiment_analyst import SentimentAnalystAgent
from agents.risk_agent.risk_manager import RiskManagerAgent
from agents.manager_agent.portfolio_manager import PortfolioManagerAgent
from hybrid_system.hybrid_predictor import HybridPredictor
from evaluation.backtest import Backtester

print("✅ Módulos importados correctamente")

## 2. Carga de Datos

In [ ]:
# Configuración
symbols = ['AAPL', 'MSFT', 'BTC-USD']
start_date = '2020-01-01'
end_date = datetime.now().strftime('%Y-%m-%d')

# Cargar datos
loader = FinancialDataLoader()
data = {}

for symbol in symbols[:2]:
    print(f"\n📊 Cargando {symbol}...")
    df = loader.download_stock_data(symbol, start_date, end_date)
    
    if not df.empty:
        processor = DataProcessor()
        df = processor.clean_data(df)
        df = processor.add_technical_indicators(df)
        data[symbol] = df
        print(f"   ✅ {len(df)} registros cargados")
        print(f"   📅 {df.index[0].date()} a {df.index[-1].date()}")
        print(f"   💰 Precio actual: ${df['Close'].iloc[-1]:.2f}")

## 3. Visualización de Datos

In [ ]:
# Visualizar precios
fig, axes = plt.subplots(len(data), 1, figsize=(14, 4*len(data)))

if len(data) == 1:
    axes = [axes]

for idx, (symbol, df) in enumerate(data.items()):
    ax = axes[idx]
    
    # Precio y medias móviles
    ax.plot(df.index, df['Close'], label='Close', linewidth=1.5)
    ax.plot(df.index, df['SMA_20'], label='SMA 20', alpha=0.7)
    ax.plot(df.index, df['SMA_50'], label='SMA 50', alpha=0.7)
    
    ax.set_title(f'{symbol} - Precio y Medias Móviles')
    ax.set_ylabel('Precio ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Detección de Patrones Chartistas

In [ ]:
# Detectar patrones para cada símbolo
pattern_results = {}

for symbol, df in data.items():
    print(f"\n{'='*60}")
    print(f"🔍 Detectando patrones en {symbol}")
    print(f"{'='*60}")
    
    detector = PatternDetector()
    patterns = detector.detect_all_patterns(df)
    
    if patterns:
        summary = detector.get_pattern_summary()
        pattern_results[symbol] = summary
        print(f"\n📊 Patrones detectados: {len(patterns)}")
        print(summary.head().to_string())
    else:
        print("   No se detectaron patrones significativos")

## 5. Entrenamiento de Modelos ML

In [ ]:
# Entrenar modelos para el primer símbolo
symbol = list(data.keys())[0]
df = data[symbol]

print(f"\n{'='*60}")
print(f"🤖 Entrenando modelos ML para {symbol}")
print(f"{'='*60}")

# Preparar características
feature_engineer = FeatureEngineer()
df_features = feature_engineer.create_features(df, prediction_horizon=5)

feature_cols = feature_engineer.feature_names
print(f"\n📊 Características creadas: {len(feature_cols)}")

# Preparar datos
X, y = feature_engineer.prepare_ml_data(df_features, feature_cols, 'target_return')

split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"📊 Datos: Train={len(X_train)}, Test={len(X_test)}")

# Entrenar modelos
ml_models = TraditionalMLModels()

ml_models.train_random_forest(X_train, y_train)
ml_models.train_xgboost(X_train, y_train)
ml_models.train_lightgbm(X_train, y_train)

# Evaluar
results = ml_models.evaluate_models(X_test, y_test)
print("\n📊 Resultados de Evaluación:")
print(results.to_string())

## 6. Sistema Multi-Agente

In [ ]:
# Crear y ejecutar sistema multi-agente
print(f"\n{'='*60}")
print(f"🤖 Ejecutando Sistema Multi-Agente")
print(f"{'='*60}")

# Crear agentes
technical_agent = TechnicalAnalystAgent()
sentiment_agent = SentimentAnalystAgent()
risk_agent = RiskManagerAgent()

comm_bus = AgentCommunicationBus()
comm_bus.register_agent(technical_agent)
comm_bus.register_agent(sentiment_agent)
comm_bus.register_agent(risk_agent)

portfolio_manager = PortfolioManagerAgent(comm_bus)
comm_bus.register_agent(portfolio_manager)

# Preparar datos
analysis_data = {
    'symbol': symbol,
    'price_data': df,
    'news': sentiment_agent.generate_mock_news(symbol, n_items=5),
    'portfolio': portfolio_manager.portfolio
}

# Ejecutar análisis
result = portfolio_manager.analyze(analysis_data)

print(f"\n✅ Análisis Completado")
print(f"   Decisión: {result.recommendation}")
print(f"   Confianza: {result.confidence:.2f}")
print(f"\n📝 Razonamiento:")
print(f"   {result.reasoning}")

## 7. Sistema Híbrido ML + LLM

In [ ]:
# Crear sistema híbrido
print(f"\n{'='*60}")
print(f"🔮 Sistema Híbrido ML + LLM")
print(f"{'='*60}")

hybrid = HybridPredictor(use_ensemble=True)

# Entrenar
hybrid.train_ml_models(df)
hybrid.train_volatility_model(df)

# Predecir
feature_cols = [col for col in df.columns if col not in ['target_return', 'target_direction', 'target_price']]
predictions = hybrid.predict(df, feature_cols)

print(f"\n📊 Predicciones:")
for model, pred in predictions.items():
    if isinstance(pred, dict):
        print(f"   {model}: {pred}")

## 8. Backtesting

In [ ]:
# Ejecutar backtest
print(f"\n{'='*60}")
print(f"📈 Backtesting")
print(f"{'='*60}")

backtester = Backtester(initial_capital=100000)

def simple_signal(data):
    """Generador de señales simple"""
    if len(data) < 20:
        return 0
    
    sma_20 = data['Close'].rolling(20).mean().iloc[-1]
    current = data['Close'].iloc[-1]
    
    if current > sma_20 * 1.02:
        return 1
    elif current < sma_20 * 0.98:
        return -1
    return 0

bt_result = backtester.run_backtest(df, simple_signal, symbol)

print(f"\n📊 Resultados:")
print(f"   Retorno Total: {bt_result.total_return*100:.2f}%")
print(f"   Sharpe Ratio: {bt_result.sharpe_ratio:.2f}")
print(f"   Max Drawdown: {bt_result.max_drawdown*100:.2f}%")
print(f"   Win Rate: {bt_result.win_rate*100:.1f}%")
print(f"   Total Trades: {bt_result.total_trades}")

## 9. Visualización de Resultados

In [ ]:
# Visualizar equity curve
plt.figure(figsize=(12, 6))
plt.plot(bt_result.equity_curve, label='Equity Curve', linewidth=2)
plt.title(f'Backtest Results - {symbol}')
plt.xlabel('Tiempo')
plt.ylabel('Valor del Portfolio ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Métricas de rendimiento
metrics = {
    'Total Return (%)': bt_result.total_return * 100,
    'Sharpe Ratio': bt_result.sharpe_ratio,
    'Max Drawdown (%)': bt_result.max_drawdown * 100,
    'Win Rate (%)': bt_result.win_rate * 100,
    'Volatility (%)': bt_result.volatility * 100,
    'Calmar Ratio': bt_result.calmar_ratio
}

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(metrics.keys(), metrics.values())
ax.set_title(f'Métricas de Rendimiento - {symbol}')
ax.set_ylabel('Valor')
plt.xticks(rotation=45, ha='right')

# Colorear barras
for bar, (key, value) in zip(bars, metrics.items()):
    if 'Return' in key or 'Sharpe' in key or 'Win' in key or 'Calmar' in key:
        bar.set_color('green' if value > 0 else 'red')
    elif 'Drawdown' in key:
        bar.set_color('red')
    else:
        bar.set_color('blue')

plt.tight_layout()
plt.show()

## 10. Conclusiones

Este notebook ha demostrado:

1. **Carga y preparación de datos**: Obtención de datos históricos de acciones y criptomonedas
2. **Detección de patrones**: Identificación automática de patrones chartistas
3. **Modelos ML**: Entrenamiento y evaluación de Random Forest, XGBoost y LightGBM
4. **Sistema Multi-Agente**: Coordinación de agentes especializados
5. **Sistema Híbrido**: Integración de ML + LLM para predicciones
6. **Backtesting**: Evaluación de estrategias con métricas de rendimiento

### Próximos pasos:
- Integrar LLMs reales (OpenAI, etc.)
- Expandir detección de patrones
- Optimizar hiperparámetros
- Implementar trading en tiempo real